# RAM Insight — เครื่องมือ Python ตรวจและเตรียมข้อมูล (Back-end)

เครื่องมือนี้เป็น "หลังบ้าน" ของเว็บ RAM Insight ทำ 4 อย่าง
1. **ทำความสะอาดไฟล์สกปรก** — ชื่อสถานีเขียนไม่เหมือนกัน (ตัวย่อ/ตัวเต็ม/อังกฤษ), ชื่อคอลัมน์ไม่ตรง, หม้อแปลงแยกหลายชีต/หลายไฟล์, วันที่หลายรูปแบบ (รวม พ.ศ.), รหัส/ชื่ออุปกรณ์พิมพ์ไม่เหมือนกัน, แถวซ้ำ/แถวว่าง/หัวตารางซ้ำ → รวมเป็นตารางมาตรฐานเดียว พร้อมบันทึกว่าแก้อะไรไปบ้าง
2. **ตรวจคุณภาพข้อมูล (Data Quality Gate)** — คัดแถวที่เวลาขาดหาย / เวลาซ่อมเสร็จก่อนเริ่มเสีย / หยุดนานเกิน 1 ปี ออก
3. **คำนวณ MTBF, MTTR, Availability (และ R, M)** — ใช้สูตรและเกณฑ์เดียวกับเว็บทุกตัว
4. **ส่งออกไฟล์ Excel ที่สะอาดแล้ว** — เอาไปอัปโหลดเข้าเว็บ RAM Insight ได้ทันที

## วิธีใช้ (กดทีละเซลล์ จากบนลงล่าง)
1. กดปุ่ม ▶ ที่เซลล์ **"ขั้นที่ 1"** และ **"ขั้นที่ 1.5"** (รอให้ขึ้นเครื่องหมายถูกสีเขียว)
2. กดปุ่ม ▶ ที่เซลล์ **"ขั้นที่ 2"** จะมีปุ่ม *Choose Files* — เลือกได้ **หลายไฟล์พร้อมกัน** (.xlsx / .csv)
3. ดูผลที่ขึ้นด้านล่าง (รายงานการทำความสะอาด + ตรวจคุณภาพ + ตารางสรุป)
4. กดปุ่ม ▶ ที่เซลล์ **"ขั้นที่ 3"** เพื่อดาวน์โหลดไฟล์ที่สะอาดแล้ว (มีชีตบอกว่าแก้อะไร และชีต "ต้องตรวจสอบ" ให้คนดูซ้ำ)

> **หลักการ:** เครื่องมือ "ล้างให้เป็นรูปแบบมาตรฐาน" แต่ **ไม่เดาสิ่งที่เดาไม่ได้** (เช่น สถานีที่ไม่รู้จัก, วันที่ที่ไม่มีอยู่จริง, เวลาเริ่ม/เสร็จกรอกสลับ) — จะส่งไปชีต "ต้องตรวจสอบ" แทน
> ถ้าไฟล์จริงใช้ชื่อสถานี/คอลัมน์ที่เครื่องมือไม่รู้จัก ให้เพิ่มชื่อในตั้งค่าด้านบนของเซลล์ 1.5 (`SUB_ALIASES_EXTRA`, `COLUMN_MAP_EXTRA`) แล้วรันใหม่

> **เรื่องข้อมูลจริง:** ถ้าไฟล์เป็นข้อมูลจริงของ กฟน. ที่เป็นความลับ การอัปโหลดขึ้น Colab คือการส่งไฟล์ไปที่เซิร์ฟเวอร์ Google
> ให้ขออนุญาตก่อน หรือใช้ข้อมูลจำลองแทน (โค้ดชุดเดียวกันรันบนเครื่องตัวเองได้ด้วย)


## ขั้นที่ 1 — โหลดกฎและสูตร (ไม่ต้องแก้อะไร)
โค้ดในเซลล์ถัดไปคือกฎชุดเดียวกับที่เขียนไว้ในหน้าเว็บ (ชื่อคอลัมน์ที่รองรับ, รูปแบบวันที่, เงื่อนไขตัดข้อมูล, สูตร R/M/A)

In [ ]:
import re, math, datetime as dt
import openpyxl

# ---- 1) ตารางชื่อคอลัมน์ที่ยอมรับ (คัดลอกจาก FIELD_ALIASES ในเว็บ ตรงตัว) ----
ALIASES = {
    'equipment_group': ['equipmentgroup','group','plant','area','site','ระบบ','กลุ่มอุปกรณ์','equipment'],
    'component': ['component','equipment','equipmentname','asset','item','part','subsystem','อุปกรณ์','ชื่ออุปกรณ์'],
    'code': ['code','id','tagno','tag','itemcode','recordid','no','failurecode','รหัส','รหัสอุปกรณ์'],
    'description': ['description','desc','detail','details','issue','problem','finding','symptom','failuremode','mode','รายละเอียด','อาการ','ลักษณะปัญหา'],
    'problem_category': ['problemcategory','category','issuetype','problemtype','type','ประเภทปัญหา','หมวดหมู่'],
    'cause': ['cause','rootcause','reason','failurecause','สาเหตุ'],
    'remedy': ['remedy','action','correctiveaction','solution','recommendation','การแก้ไข','วิธีแก้ไข'],
    'department': ['department','dept','responsible','owner','team','หน่วยงาน','แผนก'],
    'ram_classification': ['ramclassification','ramclass','ramcategory','ram'],
    'mtbf': ['mtbf','mtbfhrs','mtbfhours'],
    'mttr': ['mttr','mttrhrs','mttrhours'],
    'availability': ['availability','availabilitypct','avail','ความพร้อมใช้งาน'],
    'substation': ['substation','sub','station','สถานี','สถานีไฟฟ้า','สถานีย่อย'],
    'transformer_unit': ['transformerid','transformerunit','trno','หม้อแปลง','เลขหม้อแปลง','รหัสหม้อแปลง'],
    'mva': ['mva','ratingmva','ขนาดmva','ขนาดหม้อแปลง'],
    'failure_start': ['failurestart','startdatetime','starttime','failuretime','เวลาเริ่มขัดข้อง','วันเวลาเริ่มขัดข้อง','เวลาเริ่มต้นขัดข้อง','วันที่เริ่มขัดข้อง'],
    'repair_complete': ['repaircomplete','enddatetime','endtime','completetime','เวลาซ่อมเสร็จ','วันเวลาซ่อมเสร็จ','เวลาที่ซ่อมเสร็จ','วันที่ซ่อมเสร็จ','เวลาซ่อมแซมเสร็จ'],
}

# ---- 2) ค่ากำหนดและเกณฑ์ (เหมือนเว็บ) ----
PROFILES = {'monthly': (720, 12), 'yearly': (8760, 24)}   # (T_MISSION, T_REPAIR)
R_STD, M_STD, A_STD = 85, 50, 96                           # เกณฑ์ผ่าน (%)
MAX_DOWNTIME_HRS = 8760                                    # เกิน 1 ปี = ค่าผิดปกติ

def norm_key(k):
    return re.sub(r'[^a-z0-9\u0e01-\u0e59]', '', str(k).lower())

def js_str(v):
    if isinstance(v, float) and v.is_integer():
        v = int(v)
    return str(v).strip()

def js_parse_float(v):
    if isinstance(v, bool): return float('nan')
    if isinstance(v, (int, float)): return float(v)
    m = re.match(r'^\s*[-+]?(\d+\.?\d*|\.\d+)([eE][-+]?\d+)?', str(v))
    return float(m.group(0)) if m else float('nan')

# ---- 3) แปลงวันเวลา (เหมือน parseFlexibleDate) ----
_DMY = re.compile(r'^(\d{1,2})[/\-](\d{1,2})[/\-](\d{4})(?:[ T](\d{1,2}):(\d{2})(?::(\d{2}))?)?$')

def parse_flexible_date(val):
    if val is None or val == '':
        return None
    if isinstance(val, dt.datetime):
        return val
    if isinstance(val, dt.date):
        return dt.datetime(val.year, val.month, val.day)
    if isinstance(val, bool):
        return None
    if isinstance(val, (int, float)):            # เลขลำดับวันที่ของ Excel
        try:
            return dt.datetime(1970, 1, 1) + dt.timedelta(milliseconds=round((val - 25569) * 86400 * 1000))
        except (OverflowError, ValueError):
            return None
    s = str(val).strip()
    if not s:
        return None
    m = _DMY.match(s)
    if m:
        d, mo, y, h, mi, sec = m.groups()
        d, mo, y = int(d), int(mo), int(y)
        y += (mo - 1) // 12; mo = (mo - 1) % 12 + 1      # JS ยอมให้เดือน/วันเกินแล้วเลื่อนต่อ
        try:
            return dt.datetime(y, mo, 1) + dt.timedelta(days=d - 1, hours=int(h or 0), minutes=int(mi or 0), seconds=int(sec or 0))
        except (OverflowError, ValueError):
            return None
    try:
        return dt.datetime.fromisoformat(s.replace('Z', ''))
    except ValueError:
        return None

# ---- 4) อ่านแถว → รายการ (เหมือน mapRow / classifyEquipment / จำแนก R-A-M) ----
def normalize_sub(v):                          # เหมือน normalizeSub ในเว็บ
    t = '' if v is None else js_str(v).strip()
    if not t: return ''
    l = t.lower()
    if 'ฉลอง' in l or 'chalong' in l: return 'ฉลองกรุง'
    if 'ลาดกระบัง' in l or 'lat krabang' in l or 'ladkrabang' in l: return 'ลาดกระบัง'
    return t

def map_row(raw, sheet):
    norm = {}
    for k, v in raw.items():
        norm[norm_key(k)] = v
    out = {}
    for field, names in ALIASES.items():
        val = None
        for a in names:
            if a in norm and norm[a] is not None and norm[a] != '':
                val = norm[a]; break
        out[field] = val
    g = lambda f: js_str(out[f]) if out[f] not in (None, '', 0) else ''
    return {
        'sheet': sheet,
        'equipment_group': g('equipment_group'), 'component': g('component'),
        'substation': normalize_sub(out['substation']), 'unit': g('transformer_unit'),
        'code': g('code'), 'description': g('description'),
        'problem_category': g('problem_category') or (sheet or ''),
        'cause': g('cause'), 'remedy': g('remedy'), 'department': g('department'),
        'ram_raw': g('ram_classification'),
        'mtbf': js_parse_float(out['mtbf']) if out['mtbf'] is not None else float('nan'),
        'mttr': js_parse_float(out['mttr']) if out['mttr'] is not None else float('nan'),
        'availability_file': js_parse_float(out['availability']) if out['availability'] is not None else float('nan'),
        'failure_start': parse_flexible_date(out['failure_start']),
        'repair_complete': parse_flexible_date(out['repair_complete']),
    }

def equip_category(r):
    if r['component']:
        return r['component'].replace('_', ' ').replace('/', ' / ').strip()
    return 'ไม่ระบุ'

def parse_ram(raw):
    if not raw: return None
    low = raw.lower(); primary = None
    if 'reliability' in low: primary = 'Reliability'
    elif 'availability' in low: primary = 'Availability'
    elif 'maintainability' in low: primary = 'Maintainability'
    if not primary:
        t = raw.strip().upper()
        full = {'R': 'Reliability', 'A': 'Availability', 'M': 'Maintainability'}.get(t)
        return (full, '-', 'จากไฟล์') if full else None
    sub = re.sub(primary, '', raw, count=1, flags=re.I)
    sub = re.sub(r'^[\s/:\-()]+', '', sub); sub = re.sub(r'[)\s]+$', '', sub).strip() or '-'
    return (primary, sub, 'จากไฟล์')

def auto_ram(r):
    text = ((r['description'] or '') + ' ' + (r['cause'] or '')).lower()
    has = lambda *w: any(x in text for x in w)
    if has('รั่ว', 'leak') or (has('น้ำมัน') and has('ต่ำ', 'ระดับ')) or has('low level'):
        return ('Availability', 'รั่วซึม/ระดับต่ำ', 'อัตโนมัติ')
    if has('สนิม', 'เสื่อมสภาพ', 'เปลี่ยนสี', 'corrosion', 'rust', 'aging', 'degrad'):
        return ('Reliability', 'เสื่อมสภาพ', 'อัตโนมัติ')
    if has('รอจัดซื้อ', 'ไม่มีอะไหล่', 'อะไหล่', 'spare', 'procure', 'waiting part'):
        return ('Maintainability', 'รออะไหล่/จัดซื้อ', 'อัตโนมัติ')
    if has('trip', 'ขัดข้อง', 'ไม่ทำงาน', 'fault', 'fail', 'malfunction'):
        return ('Reliability', 'ขัดข้อง/ทำงานผิดปกติ', 'อัตโนมัติ')
    if has('ชำรุด', 'broken', 'damage', 'defect'):
        return ('Reliability', 'ชำรุด', 'อัตโนมัติ')
    return ('Maintainability', 'ไม่เข้าเงื่อนไขใด (ค่าเริ่มต้น)', 'อัตโนมัติ')

def availability_pct(r):                       # เหมือน computeAvailabilityPct
    if math.isfinite(r['mtbf']) and math.isfinite(r['mttr']) and (r['mtbf'] + r['mttr']) > 0:
        return r['mtbf'] / (r['mtbf'] + r['mttr']) * 100
    if math.isfinite(r['availability_file']):
        v = r['availability_file']
        return v * 100 if v <= 1 else v
    return float('nan')

def read_records(path):
    wb = openpyxl.load_workbook(path, data_only=True)
    recs = []
    for ws in wb.worksheets:
        rows = list(ws.iter_rows(values_only=True))
        if len(rows) < 2: continue
        head = [h if h is not None else '' for h in rows[0]]
        for row in rows[1:]:
            raw = {str(h): v for h, v in zip(head, row) if str(h) != ''}
            r = map_row(raw, ws.title)
            if not r['description'] and not r['component'] and not r['code']:
                continue
            r['equipCategory'] = equip_category(r)
            ram = parse_ram(r['ram_raw']) or auto_ram(r)
            r['ramPrimary'], r['ramSub'], r['ramSource'] = ram
            r['availabilityPct'] = availability_pct(r)
            r['raw'] = raw
            r['excluded_reason'] = ''
            r['downtimeHrs'] = float('nan')
            recs.append(r)
    return recs

# ---- 5) Data Quality Gate + MTBF/MTTR (เหมือน applyTimestampPipeline) ----
def quality_gate(recs):
    with_ts = [r for r in recs if r['failure_start'] or r['repair_complete']]
    if not with_ts:
        return {'hasTimestampData': False, 'total': 0, 'valid': 0, 'excluded': []}
    excluded, valid = [], []
    for r in with_ts:
        if not r['failure_start'] or not r['repair_complete']:
            excluded.append((r['code'], 'ข้อมูลเวลาขาดหาย (Missing time value)'))
            r['excluded_reason'] = 'ข้อมูลเวลาขาดหาย'; continue
        if r['repair_complete'] <= r['failure_start']:
            excluded.append((r['code'], 'เวลาซ่อมเสร็จเกิดก่อน/พร้อมเวลาขัดข้อง (Logical time error)'))
            r['excluded_reason'] = 'เวลาไม่สมเหตุสมผล (ซ่อมเสร็จก่อนเริ่มเสีย)'; continue
        hrs = (r['repair_complete'] - r['failure_start']).total_seconds() / 3600
        if hrs > MAX_DOWNTIME_HRS:
            excluded.append((r['code'], f'Downtime ผิดปกติ ({hrs:.0f} ชม. > 1 ปี)'))
            r['excluded_reason'] = 'Downtime ผิดปกติ (Outlier)'; continue
        r['downtimeHrs'] = hrs
        valid.append(r)
    groups = {}
    for r in valid:
        groups.setdefault((r['substation'], r['equipCategory']), []).append(r)   # แยกกลุ่มตามสถานี (ไม่มีคอลัมน์สถานี = เหมือนเดิม)
    for g in groups.values():
        g.sort(key=lambda r: r['failure_start'])
        ups = [(g[i]['failure_start'] - g[i - 1]['repair_complete']).total_seconds() / 3600 for i in range(1, len(g))]
        ups = [u for u in ups if u > 0]
        gm = sum(ups) / len(ups) if ups else float('nan')
        for r in g:
            r['mtbf'] = gm
            r['mttr'] = r['downtimeHrs']
            r['availabilityPct'] = availability_pct(r)
    return {'hasTimestampData': True, 'total': len(with_ts), 'valid': len(valid), 'excluded': excluded}

# ---- 6) สูตร R / M / A (เหมือน calcDimensions) ----
def calc_dimensions(mtbf, mttr, profile='monthly'):
    t_mission, t_repair = PROFILES[profile]
    R = math.exp(-t_mission / mtbf) * 100
    M = (1 - math.exp(-t_repair / mttr)) * 100
    A = mtbf / (mtbf + mttr) * 100
    return {'R': R, 'M': M, 'A': A, 'annualDowntime': 8760 * (1 - A / 100)}

def stats_from_occurrences(occ):               # เหมือน computeStatsFromOccurrences
    n = len(occ)
    mttr = sum(r['downtimeHrs'] for r in occ) / n if n else float('nan')
    mtbf = float('nan')
    if n >= 2:
        ups = [(occ[i]['failure_start'] - occ[i - 1]['repair_complete']).total_seconds() / 3600 for i in range(1, n)]
        ups = [u for u in ups if u > 0]
        if ups: mtbf = sum(ups) / len(ups)
    return {'mtbf': mtbf, 'mttr': mttr, 'count': n}

def summarize(recs, profile='monthly', sub=None):
    """สรุปรายประเภทอุปกรณ์ (เหมือนค่า baseline ที่หน้า What-If ใช้) — sub = เลือกเฉพาะสถานี (เหมือนปุ่มสถานีในเว็บ)"""
    cats = {}
    for r in recs:
        if sub and r['substation'] != sub: continue
        if math.isfinite(r['mtbf']) and math.isfinite(r['mttr']):
            cats.setdefault(r['equipCategory'], []).append(r)
    rows = []
    for c, rs in sorted(cats.items()):
        mtbf = sum(r['mtbf'] for r in rs) / len(rs)
        mttr = sum(r['mttr'] for r in rs) / len(rs)
        d = calc_dimensions(mtbf, mttr, profile)
        rows.append({'equipCategory': c, 'incidents': len(rs), 'MTBF_hrs': mtbf, 'MTTR_hrs': mttr,
                     'R_pct': d['R'], 'M_pct': d['M'], 'A_pct': d['A'], 'annual_downtime_hrs': d['annualDowntime'],
                     'R_pass': d['R'] >= R_STD, 'M_pass': d['M'] >= M_STD, 'A_pass': d['A'] >= A_STD})
    return rows

def code_level(recs, sub=None):
    """MTBF/MTTR รายรหัส (เหมือน computeCodeLevelStats)"""
    if sub: recs = [r for r in recs if r['substation'] == sub]
    keys = sorted({(r['equipCategory'], r['code']) for r in recs if math.isfinite(r['downtimeHrs']) and r['failure_start']})
    out = []
    for cat, code in keys:
        occ = sorted([r for r in recs if r['equipCategory'] == cat and r['code'] == code
                      and math.isfinite(r['downtimeHrs']) and r['failure_start']], key=lambda r: r['failure_start'])
        out.append({'equipCategory': cat, 'code': code, **stats_from_occurrences(occ)})
    return out


## ขั้นที่ 1.5 — โหลดตัวทำความสะอาดไฟล์ (ไม่ต้องแก้อะไร ยกเว้นเพิ่มชื่อที่ไฟล์จริงใช้)
ส่วนนี้ **ไม่มีสูตร** — ทำหน้าที่จัดรูปแบบข้อมูลก่อนส่งเข้าสูตรชุดเดียวกับเว็บ

In [ ]:
# ======================= ตัวทำความสะอาดไฟล์สกปรก (ทำงานก่อนด่านตรวจและสูตรของเว็บ) =======================
# หลักการ: "ล้างให้เป็นรูปแบบมาตรฐาน" เท่านั้น ไม่เดาข้อมูลที่เดาไม่ได้ — สิ่งที่ไม่แน่ใจจะถูกส่งไปชีต "ต้องตรวจสอบ"
# สูตร MTBF/MTTR/Availability ไม่ได้อยู่ในส่วนนี้ (ยังเป็นชุดเดียวกับเว็บที่เซลล์ขั้นที่ 1)
import os, csv, io, unicodedata
from collections import Counter
from openpyxl.utils.datetime import from_excel

# ---- ตั้งค่าที่แก้เองได้ (เพิ่มชื่อใหม่ที่ไฟล์จริงใช้ แล้วรันใหม่) ----
SUB_ALIASES_EXTRA = {}      # เช่น {'ลาดกระบัง': ['ลกบ', 'LKB1'], 'ฉลองกรุง': ['ฉก.']}
COLUMN_MAP_EXTRA = {}       # เช่น {'เวลาเกิดเหตุจริง': 'failure_start'}  (ฟิลด์: ดูคอลัมน์ "ฟิลด์" ในชีต การจับคู่คอลัมน์)
SUB_CODES = {'ลาดกระบัง': 'LKB', 'ฉลองกรุง': 'CLK'}

OUT_HEADERS = {
    'substation': 'สถานี', 'transformer_unit': 'หม้อแปลง', 'mva': 'ขนาด (MVA)', 'equipment_group': 'Equipment',
    'component': 'Component', 'code': 'รหัส', 'description': 'Description', 'problem_category': 'Problem_Category',
    'failure_start': 'เวลาเริ่มขัดข้อง', 'repair_complete': 'เวลาซ่อมเสร็จ', 'cause': 'สาเหตุ', 'remedy': 'การแก้ไข',
    'department': 'หน่วยงาน', 'ram_classification': 'RAM_Classification',
    'std_response': 'มาตรฐาน Response (ชม.)', 'std_repair': 'มาตรฐาน Repair (ชม.)',
    'mtbf': 'MTBF', 'mttr': 'MTTR', 'availability': 'Availability'}
OUT_ORDER = list(OUT_HEADERS)
CORE_FIELDS = [f for f in ALIASES]   # ฟิลด์ที่เว็บรู้จัก (จากเซลล์ขั้นที่ 1)

_BASE_SUB = {
    'ลาดกระบัง': ['ลาดกระบัง', 'ladkrabang', 'latkrabang', 'lkb', 'lkbg', 'lk', 'ลก', 'lkrabang'],
    'ฉลองกรุง': ['ฉลองกรุง', 'chalongkrung', 'chalongkrungg', 'ck', 'clk', 'ckk', 'chk', 'ฉก', 'chalong'],
}
_SUB_PREFIX = ['สถานีไฟฟ้าแรงสูง', 'สถานีไฟฟ้า', 'สถานีย่อย', 'สถานี', 'สฟ', 'substation', 'station', 'sub', 'นิคมอุตสาหกรรม', 'นิคม']
_SUB_FULL = {'ลาดกระบัง': ['ลาดกระบัง', 'ladkrabang', 'latkrabang'], 'ฉลองกรุง': ['ฉลองกรุง', 'chalong']}

def _nk(s):
    return norm_key(s)

def sub_lookup():
    d = {}
    for canon, names in _BASE_SUB.items():
        for n in names + SUB_ALIASES_EXTRA.get(canon, []):
            d[_nk(n)] = canon
    return d

def canon_sub(text):
    """คืน (ชื่อมาตรฐาน | None, ข้อความที่เจอหลายสถานีพร้อมกันไหม)"""
    k = _nk(text)
    if not k: return None, False
    lk = sub_lookup()
    if k in lk: return lk[k], False
    k2 = k
    for p in sorted(_SUB_PREFIX, key=len, reverse=True):
        pk = _nk(p)
        if k2.startswith(pk): k2 = k2[len(pk):]
    if k2 in lk: return lk[k2], False
    toks = [t for t in re.split(r'[^A-Za-z0-9ก-๙]+', str(text).lower()) if t]
    tk = {lk[_nk(t)] for t in toks if _nk(t) in lk}
    if len(tk) == 1: return next(iter(tk)), False
    hits = {c for c, ws in _SUB_FULL.items() if any(_nk(w) in k for w in ws)}
    for c, ws in SUB_ALIASES_EXTRA.items():
        if any(len(_nk(w)) >= 4 and _nk(w) in k for w in ws): hits.add(c)
    if len(hits) == 1: return next(iter(hits)), False
    if len(hits) > 1: return None, True
    return None, False

_UNIT_RE = re.compile(r'(?:tr|t|no\.?|#|หม้อแปลง(?:ลูก)?(?:ที่|เลขที่)?|ลูกที่|เครื่องที่|ตัวที่|unit)\s*[-_.]?\s*0*(\d{1,2})(?!\d)', re.I)

_UNIT_NAME_RE = re.compile(r'(?:tr|no\.?|#|หม้อแปลง(?:ลูก)?(?:ที่|เลขที่)?|ลูกที่|เครื่องที่|ตัวที่|unit)\s*[-_.]?\s*0*(\d{1,2})(?!\d)', re.I)

def parse_unit_no(text, strict=False):
    s = str(text).strip()
    if not s: return None
    if re.fullmatch(r'0*\d{1,2}', s): return int(s)
    m = (_UNIT_NAME_RE if strict else _UNIT_RE).search(s)
    if m: return int(m.group(1))
    if strict:
        m2 = re.search(r'(?:^|[\s_.\-])0*(\d{1,2})\s*$', s)
        if m2 and canon_sub(s)[0]: return int(m2.group(1))
    return None

def canon_unit(text, sub):
    n = parse_unit_no(text)
    if n is None: return None
    return (f"{SUB_CODES[sub]}-TR{n}" if sub in SUB_CODES else f"TR{n}")

_COMP_RULES = [
    ('OLTC', ['oltc', 'tapchang', 'ตัวปรับแรงดัน', 'แทป']),
    ('Oil Filter', ['oilfilter', 'เครื่องกรองน้ำมัน', 'กรองน้ำมัน']),
    ('Main Tank / Accessories', ['maintank', 'accessor', 'ตัวถัง', 'อุปกรณ์ประกอบ']),
    ('Control', ['control', 'ควบคุม', 'protection']),
]
def canon_component(text):
    k = _nk(text)
    if not k: return None
    for canon, keys in _COMP_RULES:
        if any(_nk(x) in k for x in keys): return canon
    return None

def canon_code(text):
    s = re.sub(r'\s+', '', str(text)).strip()
    m = re.fullmatch(r'([A-Za-z])[-_.]*0*(\d{1,3})', s)
    if m:
        return m.group(1).upper() + m.group(2).zfill(3)
    return None

# ---------- วันที่ ----------
_TH_DIG = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')
_MONTHS = {}
for i, names in enumerate([
    ['มกราคม', 'ม.ค.', 'มค', 'jan', 'january'], ['กุมภาพันธ์', 'ก.พ.', 'กพ', 'feb', 'february'],
    ['มีนาคม', 'มี.ค.', 'มีค', 'mar', 'march'], ['เมษายน', 'เม.ย.', 'เมย', 'apr', 'april'],
    ['พฤษภาคม', 'พ.ค.', 'พค', 'may'], ['มิถุนายน', 'มิ.ย.', 'มิย', 'jun', 'june'],
    ['กรกฎาคม', 'ก.ค.', 'กค', 'jul', 'july'], ['สิงหาคม', 'ส.ค.', 'สค', 'aug', 'august'],
    ['กันยายน', 'ก.ย.', 'กย', 'sep', 'sept', 'september'], ['ตุลาคม', 'ต.ค.', 'ตค', 'oct', 'october'],
    ['พฤศจิกายน', 'พ.ย.', 'พย', 'nov', 'november'], ['ธันวาคม', 'ธ.ค.', 'ธค', 'dec', 'december']], 1):
    for n in names: _MONTHS[_nk(n)] = i

def clean_date(v):
    """คืน (datetime | None, สถานะ, หมายเหตุ)  สถานะ: ok / fixed / check / blank / invalid"""
    if v is None or (isinstance(v, str) and not v.strip()): return None, 'blank', ''
    if isinstance(v, dt.datetime) or isinstance(v, dt.date):
        d = v if isinstance(v, dt.datetime) else dt.datetime(v.year, v.month, v.day)
        if d.year > 2400: return d.replace(year=d.year - 543), 'fixed', 'ปี พ.ศ. → ค.ศ.'
        return d, 'ok', ''
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        if 20000 < v < 80000:
            try: return from_excel(v), 'fixed', 'ตัวเลขวันที่ของ Excel → วันที่'
            except Exception: pass
        return None, 'invalid', 'ตัวเลขที่ไม่ใช่วันที่'
    s = str(v).translate(_TH_DIG).strip()
    s = s.replace('น.', ' ').replace('เวลา', ' ').strip()
    s = re.sub(r'\s+', ' ', s)
    note = []
    # ชื่อเดือนไทย/อังกฤษ
    mm = re.match(r'^(\d{1,2})\s*[-/ ]?\s*([A-Za-zก-๙.]+)\s*[-/ ]?\s*(\d{2,4})(.*)$', s)
    if mm and _nk(mm.group(2)) in _MONTHS:
        s = f"{mm.group(1)}/{_MONTHS[_nk(mm.group(2))]}/{mm.group(3)}{mm.group(4)}"
        note.append('แปลงชื่อเดือน')
    tm = r'(?:[ T,]+(\d{1,2})[:.](\d{2})(?:[:.](\d{2}))?)?'
    m = re.fullmatch(r'(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{2,4})' + tm, s)
    iso = re.fullmatch(r'(\d{4})[/\-.](\d{1,2})[/\-.](\d{1,2})' + tm, s) if not m else None
    check = False
    if m:
        d_, mo, y, hh, mi, ss = m.groups()
        d_, mo, y = int(d_), int(mo), int(y)
        if y < 100:
            y = (2500 + y - 543) if y >= 40 else 2000 + y
            note.append('ปี 2 หลัก (ตีความ)'); check = True
        if mo > 12 and d_ <= 12:
            d_, mo = mo, d_; note.append('สลับ วัน/เดือน'); check = True
    elif iso:
        y, mo, d_, hh, mi, ss = iso.groups()
        y, mo, d_ = int(y), int(mo), int(d_)
    else:
        return None, 'invalid', 'อ่านรูปแบบวันที่ไม่ออก'
    if y > 2400: y -= 543; note.append('ปี พ.ศ. → ค.ศ.')
    try:
        d = dt.datetime(y, mo, d_, int(hh or 0), int(mi or 0), int(ss or 0))
    except ValueError:
        return None, 'invalid', 'วันที่หรือเวลาไม่มีอยู่จริง'
    if hh is None: note.append('ไม่มีเวลา ใช้ 00:00'); check = True
    if note: return d, ('check' if check else 'fixed'), ', '.join(note)
    return d, 'ok', ''

def fmt_dt(d):
    return d.strftime('%d/%m/%Y %H:%M:%S') if d.second else d.strftime('%d/%m/%Y %H:%M')

# ---------- หัวคอลัมน์ ----------
_END_W = ['เสร็จ', 'สิ้นสุด', 'คืนระบบ', 'กลับมา', 'จบ', 'end', 'finish', 'complete', 'restor', 'repaired', 'close', 'แก้ไขแล้ว', 'back']
_START_W = ['เริ่ม', 'เกิดเหตุ', 'เกิดขัดข้อง', 'เวลาขัดข้อง', 'เวลาเสีย', 'เสียเมื่อ', 'แจ้ง', 'start', 'begin', 'outage', 'failuretime', 'faulttime', 'downtime']

def fuzzy_field(header):
    k = _nk(header)
    if not k: return None
    if k in COLUMN_MAP_EXTRA_N: return COLUMN_MAP_EXTRA_N[k]
    hrs = ('ชม' in k) or ('hrs' in k) or ('hour' in k)
    if 'response' in k or 'เข้าไปดำเนินการ' in k: return 'std_response'
    if hrs and ('repair' in k or 'แก้ไข' in k or 'ซ่อม' in k): return 'std_repair'
    if any(x in k for x in _END_W) and not hrs: return 'repair_complete'
    if any(x in k for x in _START_W) and not hrs: return 'failure_start'
    if k in ('tr', 'trno', 'trid', 'transformer', 'transformerno', 'หม้อแปลง', 'หม้อแปลงที่', 'เลขหม้อแปลง', 'unit', 'หมายเลขหม้อแปลง'): return 'transformer_unit'
    if 'mva' in k: return 'mva'
    if 'สถานี' in k or 'substation' in k or k == 'station' or 'สฟ' in k: return 'substation'
    if ('ประเภท' in k and 'ปัญหา' in k) or 'problemcategory' in k or 'problemtype' in k: return 'problem_category'
    if 'รหัส' in k or 'code' in k: return 'code'
    if any(x in k for x in ['รายละเอียด', 'อาการ', 'description', 'detail', 'ลักษณะ']): return 'description'
    if any(x in k for x in ['สาเหตุ', 'cause', 'reason']): return 'cause'
    if any(x in k for x in ['วิธีซ่อม', 'การแก้ไข', 'แก้ไข', 'remedy', 'solution', 'corrective']): return 'remedy'
    if any(x in k for x in ['หน่วยงาน', 'แผนก', 'ทีม', 'department', 'team', 'dept', 'ผู้รับผิดชอบ', 'responsible']): return 'department'
    if any(x in k for x in ['component', 'ชื่ออุปกรณ์', 'อุปกรณ์', 'ส่วนประกอบ']): return 'component'
    if 'ram' in k: return 'ram_classification'
    return None

def _exact_field(header):
    k = _nk(header)
    return [f for f in CORE_FIELDS if k in ALIASES[f]]

COLUMN_MAP_EXTRA_N = {}

# ---------- อ่านไฟล์ ----------
_BAD = {'#n/a', '#ref!', '#value!', '#div/0!', '#name?', '#null!', 'n/a', 'na', 'null', 'none', 'nan', '-', '—', '–', '--'}
def clean_cell(v):
    if v is None: return None
    if isinstance(v, str):
        s = unicodedata.normalize('NFC', v).replace(' ', ' ').replace('​', '').replace('﻿', '')
        s = re.sub(r'\s+', ' ', s).strip()
        if s.lower() in _BAD: return None
        return s if s else None
    return v

def read_sheets(path, display_name=None):
    """คืน [(ชื่อชีต, [(เลขแถว, [ค่าในแถว])])] — เติมค่าจากเซลล์ที่ merge ให้ทุกช่อง"""
    out = []
    low = path.lower()
    if low.endswith('.csv'):
        raw = open(path, 'rb').read()
        for enc in ('utf-8-sig', 'cp874', 'utf-16'):
            try: text = raw.decode(enc); break
            except Exception: text = None
        if text is None: raise ValueError('อ่านไฟล์ CSV ไม่ได้ (ลองบันทึกเป็น .xlsx)')
        rows = [(i + 1, row) for i, row in enumerate(csv.reader(io.StringIO(text)))]
        return [(os.path.splitext(os.path.basename(display_name or path))[0], rows)]
    if low.endswith('.xls'):
        raise ValueError('ไฟล์ .xls รุ่นเก่า อ่านไม่ได้ — เปิดใน Excel แล้ว Save As เป็น .xlsx ก่อน')
    wb = openpyxl.load_workbook(path, data_only=True)
    for ws in wb.worksheets:
        merged = {}
        for rng in ws.merged_cells.ranges:
            tl = ws.cell(rng.min_row, rng.min_col).value
            for r in range(rng.min_row, rng.max_row + 1):
                for c in range(rng.min_col, rng.max_col + 1):
                    merged[(r, c)] = tl
        rows = []
        for r_i, row in enumerate(ws.iter_rows(), 1):
            vals = []
            for c in row:
                v = merged.get((c.row, c.column), c.value)
                vals.append(v)
            rows.append((r_i, vals))
        out.append((ws.title, rows))
    return out

def find_header(rows):
    best, best_i = 0, 0
    for i, (rn, vals) in enumerate(rows[:30]):
        score = 0
        for v in vals:
            if isinstance(v, str) and v.strip() and (_exact_field(v) or fuzzy_field(v)): score += 1
        if score > best: best, best_i = score, i
    return (best_i if best >= 3 else 0), best

# ---------- ตัวหลัก ----------
def clean_workbook_files(paths):
    """paths: [(path, ชื่อไฟล์ที่ผู้ใช้เห็น)]  → dict ผลลัพธ์"""
    global COLUMN_MAP_EXTRA_N
    COLUMN_MAP_EXTRA_N = {_nk(k): v for k, v in COLUMN_MAP_EXTRA.items()}
    rows_out, excluded, log, review, mapping, unused = [], [], [], [], [], []
    stats = Counter()

    def L(f, sh, rn, field, before, after, rule):
        log.append({'ไฟล์': f, 'ชีต': sh, 'แถวใน Excel': rn, 'ฟิลด์': field, 'ก่อน': '' if before is None else str(before), 'หลัง': '' if after is None else str(after), 'กฎที่ใช้': rule})
        stats[rule] += 1
    def R(level, f, sh, rn, field, val, issue):
        review.append({'ระดับ': level, 'ไฟล์': f, 'ชีต': sh, 'แถวใน Excel': rn, 'ฟิลด์': field, 'ค่า': '' if val is None else str(val), 'ประเด็น': issue})
    def X(f, sh, rn, reason, std):
        d = {'ไฟล์': f, 'ชีต': sh, 'แถวใน Excel': rn, 'เหตุผลที่ตัดออก': reason}
        d.update({OUT_HEADERS[k]: ('' if v is None else v) for k, v in std.items() if k in OUT_HEADERS and v not in (None, '')})
        excluded.append(d); stats['ตัดออก: ' + reason] += 1

    for path, disp in paths:
        fname = os.path.basename(disp or path)
        f_sub, _ = canon_sub(os.path.splitext(fname)[0])
        f_unit_no = parse_unit_no(os.path.splitext(fname)[0], strict=True)
        for sheet, rows in read_sheets(path, disp):
            if not rows: continue
            h_i, h_score = find_header(rows)
            if h_score < 3:
                R('ข้อมูล', fname, sheet, 1, 'ชีต', '', 'ข้ามชีตนี้ — ไม่พบหัวตารางที่รู้จัก (ไม่ใช่ตารางข้อมูลเหตุขัดข้อง?)')
                continue
            if h_i > 0: L(fname, sheet, rows[h_i][0], 'หัวตาราง', f'แถวหัวเรื่อง {h_i} แถวด้านบน', 'ข้าม', 'ข้ามแถวหัวเรื่องก่อนหัวตาราง')
            head_rn, head_vals = rows[h_i]
            headers = [(str(v).strip() if v is not None else '') for v in head_vals]
            # จับคู่คอลัมน์: ชื่อตรงกับของเว็บ = ใช้ตามเดิม, ที่เหลือจับแบบคำสำคัญ
            exact_have = set(); hdr_field = {}
            for j, h in enumerate(headers):
                if not h: continue
                ef = _exact_field(h)
                if ef:
                    exact_have.update(ef)
            strong_comp = any(_nk(h) in (set(ALIASES['component']) - {'equipment'}) for h in headers if h)
            if not strong_comp: exact_have.discard('component')
            for j, h in enumerate(headers):
                if not h: continue
                ef = _exact_field(h)
                if ef:
                    mapping.append({'ไฟล์': fname, 'ชีต': sheet, 'หัวคอลัมน์ในไฟล์': h, 'ฟิลด์': ef[0], 'วิธีจับคู่': 'ชื่อตรงกับที่เว็บรู้จัก'}); continue
                ff = fuzzy_field(h)
                if ff and ff not in exact_have and ff not in hdr_field.values():
                    hdr_field[j] = ff
                    mapping.append({'ไฟล์': fname, 'ชีต': sheet, 'หัวคอลัมน์ในไฟล์': h, 'ฟิลด์': ff, 'วิธีจับคู่': 'จับคู่จากคำสำคัญ (ตรวจดูด้วย)'})
                    stats['จับคู่คอลัมน์จากคำสำคัญ'] += 1
                else:
                    unused.append({'ไฟล์': fname, 'ชีต': sheet, 'หัวคอลัมน์ในไฟล์': h, 'หมายเหตุ': 'ไม่ได้ใช้ (ไม่รู้จัก หรือซ้ำกับคอลัมน์อื่น)'})
            found = exact_have | set(hdr_field.values())
            for need, th in [('code', 'รหัส'), ('failure_start', 'เวลาเริ่มขัดข้อง'), ('repair_complete', 'เวลาซ่อมเสร็จ')]:
                if need not in found:
                    R('สำคัญ', fname, sheet, head_rn, th, '', f'ไม่พบคอลัมน์ "{th}" ในชีตนี้ — คำนวณ MTBF/MTTR ไม่ได้ถ้าไม่มี')
            hdr_norm = [_nk(h) for h in headers]
            sh_sub, _ = canon_sub(sheet)
            sh_unit_no = parse_unit_no(sheet, strict=True)
            last_hdr = None
            for rn, vals in rows[h_i + 1:]:
                cells = [clean_cell(v) for v in vals]
                if not any(c is not None for c in cells): continue
                if [(_nk(c) if isinstance(c, str) else '') for c in cells[:len(hdr_norm)]] == hdr_norm:
                    L(fname, sheet, rn, 'แถว', 'หัวตารางซ้ำกลางข้อมูล', 'ลบ', 'ลบแถวหัวตารางที่ซ้ำ'); continue
                norm = {}
                for j, c in enumerate(cells):
                    if j < len(hdr_norm) and hdr_norm[j]: norm.setdefault(hdr_norm[j], c)
                std = {}
                for fld in OUT_HEADERS:
                    val = None
                    if fld == 'component':
                        for j, ff in hdr_field.items():
                            if ff == fld and j < len(cells) and cells[j] is not None: val = cells[j]; break
                    for a in ([] if val is not None else ALIASES.get(fld, [])):
                        if a in norm and norm[a] is not None: val = norm[a]; break
                    if val is None:
                        for j, ff in hdr_field.items():
                            if ff == fld and j < len(cells) and cells[j] is not None: val = cells[j]; break
                    std[fld] = val
                if not std['description'] and not std['component'] and not std['code']:
                    continue
                # --- รหัส ---
                if std['code'] is not None:
                    c0 = std['code']; c1 = canon_code(c0)
                    if c1 and c1 != c0: L(fname, sheet, rn, 'รหัส', c0, c1, 'จัดรูปแบบรหัส'); std['code'] = c1
                    elif not c1: R('ตรวจ', fname, sheet, rn, 'รหัส', c0, 'รูปแบบรหัสไม่ปกติ (ไม่ใช่ตัวอักษร+เลข เช่น T001)')
                # --- ประเภทอุปกรณ์ ---
                if std['component'] is not None:
                    c0 = std['component']; c1 = canon_component(c0)
                    if c1 and c1 != c0: L(fname, sheet, rn, 'Component', c0, c1, 'รวมชื่อประเภทอุปกรณ์ให้เป็นชื่อเดียวกัน'); std['component'] = c1
                    elif not c1: R('ข้อมูล', fname, sheet, rn, 'Component', c0, 'ประเภทอุปกรณ์ที่ไม่อยู่ในชุดหลัก 4 ประเภท (คงไว้ตามเดิม)')
                # --- สถานี ---
                s0 = std['substation']; sub = None
                if s0 is not None:
                    sub, multi = canon_sub(s0)
                    if sub:
                        if sub != s0: L(fname, sheet, rn, 'สถานี', s0, sub, 'รวมชื่อสถานีให้เป็นชื่อมาตรฐาน')
                    else:
                        R('สำคัญ', fname, sheet, rn, 'สถานี', s0, 'พบหลายสถานีในช่องเดียว' if multi else 'ไม่รู้จักชื่อสถานีนี้ (เพิ่มชื่อได้ที่ SUB_ALIASES_EXTRA) — คงไว้ตามเดิม')
                        sub = str(s0)
                else:
                    sub = sh_sub or f_sub
                    if sub: L(fname, sheet, rn, 'สถานี', '', sub, 'เติมสถานีจากชื่อชีต/ไฟล์')
                std['substation'] = sub
                # --- หม้อแปลง ---
                u0 = std['transformer_unit']; unit = None
                if u0 is not None:
                    unit = canon_unit(u0, sub)
                    if unit:
                        if unit != u0: L(fname, sheet, rn, 'หม้อแปลง', u0, unit, 'จัดรูปแบบชื่อหม้อแปลง')
                    else:
                        R('ตรวจ', fname, sheet, rn, 'หม้อแปลง', u0, 'อ่านเลขหม้อแปลงไม่ออก — คงไว้ตามเดิม'); unit = str(u0)
                else:
                    n_ = sh_unit_no if sh_unit_no is not None else f_unit_no
                    if n_ is not None:
                        unit = canon_unit(f'TR{n_}', sub); L(fname, sheet, rn, 'หม้อแปลง', '', unit, 'เติมเลขหม้อแปลงจากชื่อชีต/ไฟล์')
                std['transformer_unit'] = unit
                # --- ขนาด MVA ---
                if std['mva'] is not None:
                    m_ = re.search(r'\d+(?:\.\d+)?', str(std['mva']).replace(',', ''))
                    if m_:
                        val = float(m_.group()); val = int(val) if val == int(val) else val
                        if str(std['mva']) != str(val) and not isinstance(std['mva'], (int, float)): L(fname, sheet, rn, 'ขนาด (MVA)', std['mva'], val, 'ดึงตัวเลขขนาด MVA')
                        std['mva'] = val
                    else:
                        R('ตรวจ', fname, sheet, rn, 'ขนาด (MVA)', std['mva'], 'อ่านขนาดไม่ออก'); std['mva'] = None
                # --- วันที่เวลา ---
                bad = None; ds = {}
                for fld, th in (('failure_start', 'เวลาเริ่มขัดข้อง'), ('repair_complete', 'เวลาซ่อมเสร็จ')):
                    v0 = std[fld]
                    d, st, note = clean_date(v0)
                    ds[fld] = d
                    if st == 'blank': continue
                    if st == 'invalid': bad = bad or f'{th}: {note} ("{v0}")'; continue
                    new = fmt_dt(d)
                    if st in ('fixed', 'check') or (isinstance(v0, str) and new != v0):
                        if str(v0) != new: L(fname, sheet, rn, th, v0, new, 'จัดรูปแบบวันที่เวลา' + (f' ({note})' if note else ''))
                    if st == 'check': R('ตรวจ', fname, sheet, rn, th, v0, f'ตีความวันที่: {note} → {new}')
                    std[fld] = new
                if bad:
                    X(fname, sheet, rn, bad, std); continue
                if ds.get('failure_start') and ds.get('repair_complete') and ds['repair_complete'] <= ds['failure_start']:
                    R('สำคัญ', fname, sheet, rn, 'เวลา', f"{std['failure_start']} → {std['repair_complete']}", 'เวลาซ่อมเสร็จไม่หลังเวลาเริ่มขัดข้อง (อาจกรอกสลับกัน) — ด่านตรวจของเว็บจะตัดแถวนี้ออก ไม่ได้แก้ให้เอง')
                std['_src'] = (fname, sheet, rn); std['_dt'] = ds
                rows_out.append(std)
    # --- แถวซ้ำ ---
    seen = {}; kept = []
    for r in rows_out:
        key = (r['substation'], r['transformer_unit'], r['code'], r['failure_start'], r['repair_complete'], r['description'])
        if r['failure_start'] and key in seen:
            f, sh, rn = r['_src']
            X(f, sh, rn, f'แถวซ้ำกับแถว {seen[key][2]} ({seen[key][0]}/{seen[key][1]})', r); continue
        if r['failure_start']: seen[key] = r['_src']
        kept.append(r)
    rows_out = kept
    # --- เหตุซ้อนทับในหม้อแปลงลูกเดียวกัน ---
    byu = {}
    for r in rows_out:
        if r['transformer_unit'] and r['_dt'].get('failure_start') and r['_dt'].get('repair_complete'):
            byu.setdefault((r['substation'], r['transformer_unit']), []).append(r)
    for k, rs in byu.items():
        rs.sort(key=lambda r: r['_dt']['failure_start'])
        for a, b in zip(rs, rs[1:]):
            if b['_dt']['failure_start'] < a['_dt']['repair_complete']:
                f, sh, rn = b['_src']
                R('ตรวจ', f, sh, rn, 'เวลา', f"{b['failure_start']}", f"หม้อแปลง {k[1]} มีเหตุซ้อนกัน (เหตุก่อนหน้า {a['code']} ซ่อมเสร็จ {a['repair_complete']}) — ตรวจว่ากรอกซ้ำหรือผิดลูก")
    return {'rows': rows_out, 'excluded': excluded, 'log': log, 'review': review, 'mapping': mapping, 'unused': unused, 'stats': stats}

def std_row_to_raw(r):
    """แปลงแถวที่ล้างแล้วเป็น dict หัวคอลัมน์มาตรฐาน (ใช้ส่งต่อให้ map_row ของเว็บ/Python และเขียนลงไฟล์)"""
    out = {}
    for k in OUT_ORDER:
        v = r.get(k)
        if v is None: continue
        out[OUT_HEADERS[k]] = v
    return out

def records_from_clean(clean, sheet='ข้อมูลสะอาด'):
    recs = []
    for r in clean['rows']:
        raw = std_row_to_raw(r)
        rec = map_row(raw, sheet)
        if not rec['description'] and not rec['component'] and not rec['code']: continue
        rec['equipCategory'] = equip_category(rec)
        ram = parse_ram(rec['ram_raw']) or auto_ram(rec)
        rec['ramPrimary'], rec['ramSub'], rec['ramSource'] = ram
        rec['availabilityPct'] = availability_pct(rec)
        rec['raw'] = raw; rec['excluded_reason'] = ''; rec['downtimeHrs'] = float('nan'); rec['src'] = r['_src']
        recs.append(rec)
    return recs


## ขั้นที่ 2 — เลือกไฟล์ (หลายไฟล์ได้) แล้วให้ระบบล้าง + ตรวจ + คำนวณ

In [ ]:
# ขั้นที่ 2 — เลือกไฟล์ (หลายไฟล์ได้) → ทำความสะอาด → ตรวจ → คำนวณ
import pandas as pd, os
from collections import Counter
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30)

PROFILE = 'monthly'   # 'monthly' = ภารกิจ 720 ชม. / ซ่อม 12 ชม.   |   'yearly' = 8,760 ชม. / 24 ชม.  (เหมือนปุ่มในหน้า What-If)

try:
    from google.colab import files
    up = files.upload()
    for name, data in up.items():
        pass                       # Colab เก็บไฟล์ไว้ในโฟลเดอร์ปัจจุบันตามชื่อเดิมอยู่แล้ว
    PATHS = [(n, n) for n in up.keys()]
except ImportError:                      # รันบนเครื่องตัวเอง
    PATHS = [(p.strip(), os.path.basename(p.strip())) for p in input('พิมพ์ชื่อไฟล์ (หลายไฟล์คั่นด้วย ,) : ').split(',') if p.strip()]

clean = clean_workbook_files(PATHS)
records = records_from_clean(clean)
gate = quality_gate(records)

print(f'อ่านไฟล์ {len(PATHS)} ไฟล์ → ได้ข้อมูลที่ล้างแล้ว {len(records)} รายการ  (ตัดออกตั้งแต่ขั้นล้าง {len(clean["excluded"])} แถว)')
print('\n== รายงานการทำความสะอาด (แก้อะไรไปกี่ครั้ง) ==')
for rule, n in clean['stats'].most_common():
    print(f'  - {rule}: {n}')
rv = Counter(x['ระดับ'] for x in clean['review'])
print(f"\n== ต้องตรวจสอบด้วยตา: {sum(v for k, v in rv.items() if k != 'ข้อมูล')} ประเด็น (ดูรายละเอียดในชีต 'ต้องตรวจสอบ' ของไฟล์ที่ส่งออก) ==")
for x in [x for x in clean['review'] if x['ระดับ'] != 'ข้อมูล'][:15]:
    print(f"  [{x['ระดับ']}] {x['ไฟล์']} / {x['ชีต']} / แถว {x['แถวใน Excel']}: {x['ประเด็น']}")

print('\n== ตรวจคุณภาพข้อมูล (ด่านเดียวกับเว็บ) ==')
if not gate['hasTimestampData']:
    print('ไม่พบข้อมูลเวลา (เวลาเริ่มขัดข้อง / เวลาซ่อมเสร็จ) — คำนวณ MTBF/MTTR จากเวลาไม่ได้')
else:
    print(f"มีข้อมูลเวลา {gate['total']} รายการ → ใช้คำนวณได้จริง {gate['valid']} รายการ")
    if gate['excluded']:
        print(f"ตัดออก {len(gate['excluded'])} รายการ:")
        for reason, n in Counter(e[1] for e in gate['excluded']).items():
            print(f'  - {reason}: {n} รายการ')
    else:
        print('ไม่มีรายการที่ถูกตัดออก (ข้อมูลผ่านเกณฑ์ทั้งหมด)')

summary = pd.DataFrame(summarize(records, PROFILE))
print('\n== สรุปรายประเภทอุปกรณ์ (ตัวเลขชุดนี้ต้องตรงกับหน้า What-If เมื่อเลือกประเภทเดียวกัน) ==')
display(summary.round(2)) if 'display' in globals() else print(summary.round(2))

# สรุปแยกสถานี — ตรงกับปุ่มเลือกสถานีในเว็บ
by_sub = pd.DataFrame()
subs = sorted({r['substation'] for r in records if r['substation']})
if subs:
    parts = []
    for sb in subs:
        d = pd.DataFrame(summarize(records, PROFILE, sub=sb)); d.insert(0, 'สถานี', sb); parts.append(d)
    by_sub = pd.concat(parts, ignore_index=True)
    print('\n== สรุปแยกสถานี (ตรงกับปุ่ม ลาดกระบัง / ฉลองกรุง ในเว็บ) ==')
    display(by_sub.round(2)) if 'display' in globals() else print(by_sub.round(2))

# สรุปแยกหม้อแปลงรายลูก (ข้อมูลเสริมสำหรับดูเทียบกัน — เว็บคำนวณ MTBF รวมทั้งสถานี)
by_unit = pd.DataFrame()
_g = {}
for r in records:
    if r['failure_start'] and r['repair_complete'] and r['downtimeHrs'] == r['downtimeHrs'] and r.get('transformer_unit'):
        _g.setdefault((r['substation'] or '', r['transformer_unit'], r['equipCategory']), []).append(r)
_rows = []
for (sb, un, cat), occ in sorted(_g.items()):
    occ.sort(key=lambda r: r['failure_start'])
    st = stats_from_occurrences(occ)
    _rows.append({'สถานี': sb, 'หม้อแปลง': un, 'equipCategory': cat, 'incidents': st['count'], 'MTBF_hrs': st['mtbf'], 'MTTR_hrs': st['mttr']})
by_unit = pd.DataFrame(_rows)
if len(by_unit):
    print('\n== สรุปแยกหม้อแปลงรายลูก ==')
    display(by_unit.round(2)) if 'display' in globals() else print(by_unit.round(2))

codes = pd.DataFrame(code_level(records))
print('\n== สรุปรายรหัสปัญหา (MTBF จะมีค่าเมื่อรหัสนั้นเกิดซ้ำตั้งแต่ 2 ครั้ง) ==')
display(codes.round(2)) if 'display' in globals() else print(codes.round(2))


## ขั้นที่ 3 — ดาวน์โหลดไฟล์ที่สะอาดแล้ว

In [ ]:
# ขั้นที่ 3 — ส่งออกไฟล์
OUT = 'RAM_Cleaned_Failure_Log.xlsx'

ok_rows = []
gate_bad = []
for r in records:
    row = dict(r['raw'])
    if r['excluded_reason']:
        row['เหตุผลที่ตัดออก'] = 'ด่านตรวจเว็บ: ' + str(r['excluded_reason'])
        gate_bad.append(row)
    else:
        ok_rows.append(row)
bad_rows = pd.DataFrame(clean['excluded'] + gate_bad)

with pd.ExcelWriter(OUT, engine='openpyxl') as xw:
    pd.DataFrame(ok_rows).to_excel(xw, sheet_name='ข้อมูลสะอาด', index=False)          # อัปโหลดเข้าเว็บได้
    bad_rows.to_excel(xw, sheet_name='แถวที่ถูกตัดออก', index=False)
    pd.DataFrame(clean['review']).to_excel(xw, sheet_name='ต้องตรวจสอบ', index=False)
    pd.DataFrame(clean['log']).to_excel(xw, sheet_name='รายงานการทำความสะอาด', index=False)
    pd.concat([pd.DataFrame(clean['mapping']), pd.DataFrame(clean['unused'])], ignore_index=True).to_excel(xw, sheet_name='การจับคู่คอลัมน์', index=False)
    summary.to_excel(xw, sheet_name='สรุปรายประเภท', index=False)
    codes.to_excel(xw, sheet_name='สรุปรายรหัส', index=False)
    if len(by_sub): by_sub.to_excel(xw, sheet_name='สรุปแยกสถานี', index=False)
    if len(by_unit): by_unit.to_excel(xw, sheet_name='สรุปแยกหม้อแปลง', index=False)

print(f'สร้างไฟล์ {OUT} แล้ว  (สะอาด {len(ok_rows)} แถว / ตัดออก {len(bad_rows)} แถว / ต้องตรวจสอบ {len(clean["review"])} รายการ)')
print('ชีต "ข้อมูลสะอาด" คือไฟล์ที่นำไปอัปโหลดในเว็บ RAM Insight ได้เลย — ให้ดูชีต "ต้องตรวจสอบ" ก่อนใช้ข้อมูลจริง')
try:
    from google.colab import files
    files.download(OUT)
except ImportError:
    pass
